# Optimized Bayesian Spatial Modeling of Business Success

This notebook implements a performance-tuned Bayesian Hierarchical model with Spatial ICAR effects for São Paulo state.

### 🛠️ Setup: Install Dependencies
Run this cell if you haven't installed the needed libraries yet.

In [ ]:
%pip install pymc arviz libpysal esda spreg statsmodels pytensor seaborn matplotlib numpy pandas geopandas shapely

### ⚠️ Performance Note
If you see a warning about `g++ not available`, the model will run slower in "Python-only" mode.
**To fix (recommended):** Run `conda install -c conda-forge m2w64-toolchain` in your terminal.

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely import wkt
import matplotlib.pyplot as plt
import seaborn as sns
import arviz as az
import libpysal as lps
from esda.moran import Moran
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pymc as pm
import pytensor.tensor as pt
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="viridis")

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


## 1. Data Preparation

We load only needed columns and filter for SP to save RAM.

In [ ]:
DATA_PATH = Path("../../data/processed/final_integrated_dataset_enhanced.csv")

cols_to_use = [
    'ibge_code', 'state', 'active', 'failed', 'geometry', 
    'HDI_income', 'road_density_km_km2', 'ndbi', 'GDP_per_capita'
]

df_raw = pd.read_csv(DATA_PATH, usecols=cols_to_use)
df_sp = df_raw[df_raw['state'] == 'SP'].copy()
df_sp['geometry'] = df_sp['geometry'].apply(wkt.loads)
gdf_sp = gpd.GeoDataFrame(df_sp, geometry='geometry', crs="EPSG:4326")

# Pre-processing: Remove invalid rows
gdf_sp['total'] = gdf_sp['active'] + gdf_sp['failed']
gdf_sp = gdf_sp[gdf_sp['total'] > 0].reset_index(drop=True)

# Fill NaNs with median to avoid numerical issues
predictors = ['HDI_income', 'road_density_km_km2', 'ndbi', 'GDP_per_capita']
for col in predictors:
    gdf_sp[col] = gdf_sp[col].fillna(gdf_sp[col].median())

# Drop islands
w = lps.weights.Queen.from_dataframe(gdf_sp)
if w.islands:
    gdf_sp = gdf_sp.drop(gdf_sp.index[w.islands]).reset_index(drop=True)
    w = lps.weights.Queen.from_dataframe(gdf_sp)

print(f"Municipalities for model: {len(gdf_sp)}")

## 2. Variable Scaling

Z-score standardization.

In [ ]:
for col in predictors:
    std = gdf_sp[col].std()
    if std == 0: std = 1.0 # Avoid division by zero
    gdf_sp[f'z_{col}'] = (gdf_sp[col] - gdf_sp[col].mean()) / std

## 3. Robust Bayesian ICAR Model

In [ ]:
def get_edge_indices(w):
    adj = w.neighbors
    id_map = {id: i for i, id in enumerate(w.id_order)}
    n1, n2 = [], []
    for node, neighbors in adj.items():
        for neighbor in neighbors:
            if id_map[node] < id_map[neighbor]:
                n1.append(id_map[node])
                n2.append(id_map[neighbor])
    return np.array(n1), np.array(n2)

def run_spatial_model(gdf, w_obj, samples=1000, tune=1000):
    z_cols = [f'z_{c}' for c in predictors]
    X = gdf[z_cols].values.astype('float32')
    successes = gdf['active'].values.astype('int32')
    total_count = gdf['total'].values.astype('int32')
    n_muni = len(gdf)
    n1, n2 = get_edge_indices(w_obj)

    with pm.Model() as model:
        # Priors
        alpha = pm.Normal("alpha", mu=-1, sigma=2) # Centered on -1 (typical logit for medium success)
        beta = pm.Normal("beta", mu=0, sigma=1, shape=X.shape[1])
        sigma_phi = pm.Exponential("sigma_phi", lam=2.0)
        phi_raw = pm.Normal("phi_raw", mu=0, sigma=1, shape=n_muni)
        phi = pm.Deterministic("phi", (phi_raw - pt.mean(phi_raw)) * sigma_phi)
        
        # ICAR term
        pm.Potential("spatial_prior", -0.5 * pt.sum(pt.sqr(phi[n1] - phi[n2])))
        
        # Likelihood
        logit_p = alpha + pt.dot(X, beta) + phi
        y = pm.Binomial("y_obs", n=total_count, logit_p=logit_p, observed=successes)
        
        # Efficient sampling
        trace = pm.sample(samples, tune=tune, target_accept=0.9, chains=2, cores=1, random_seed=42)
        pm.sample_posterior_predictive(trace, extend_inferencedata=True)
        
    return trace

## 4. Execution and Visualization

In [ ]:
# Run Model
trace = run_spatial_model(gdf_sp, w, samples=500, tune=500)

# 1. Plot Results
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
az.plot_forest(trace, var_names=['beta'], combined=True, ax=ax[0])
ax[0].set_yticklabels(reversed(predictors))
ax[0].set_title("Impact of Predictors (Log-Odds Scale)")
ax[0].axvline(0, color='black', alpha=0.3, ls='--')

y_pred = trace.posterior_predictive['y_obs'].mean(dim=['chain', 'draw'])
ax[1].scatter(gdf_sp['active'], y_pred, alpha=0.4)
ax[1].plot([0, max(gdf_sp['active'])], [0, max(gdf_sp['active'])], 'r--')
ax[1].set_title("Predicted vs Observed Active Businesses")
plt.show()

# 2. Spatial Map
gdf_sp['phi_mean'] = trace.posterior['phi'].mean(dim=['chain', 'draw']).values
fig, ax = plt.subplots(figsize=(10, 8))
gdf_sp.plot(column='phi_mean', cmap='RdBu_r', legend=True, ax=ax)
ax.set_title("Spatial Random Effect (phi) - Unexplained Variation")
plt.axis('off')
plt.show()